# ML Engineer quickstart

**Dataset:** `esophageal_posts` by default; Banking or your own file also work. **How long:** about five minutes once runtimes are installed.

Checks the database Python runtime, MLflow and the shared model directory, then loads and describes the data.

**Run first:** `exakit udf-use default` (installs the ML runtime; a container install can restart the database).

## Start a workflow

One ID links data, checks and model runs. To continue the Data Scientist's log, start the kernel with `EXAKIT_WORKFLOW_ID`.

In [ ]:
import os
import exakit.nb as nb

wid = nb.workflow("ml-engineer-quickstart", resume=os.environ.get("EXAKIT_WORKFLOW_ID"))
print(f"Every step below is logged under it: exakit workflow show {wid}")

## Pick the database Python runtime

Training runs as a UDF inside Exasol, with its own packages.

- `PYTHON3_ML` (`exakit udf-use default`): preferred, has the ML packages.
- `PYTHON3` (`exakit udf-setup`): the official alternative.
- `EXAKIT_UDF_CONTAINER` picks another installed alias.

In [ ]:
CONTAINER = nb.ml_train.udf_container()   # or "PYTHON3_ML", "PYTHON3", or the alias of a container you installed
print(f"container alias: {CONTAINER}")

## Connect

Uses the kit's saved connection, then lists the schemas. No password in the notebook.

In [ ]:
import exakit.nb as nb

conn = nb.connect()
nb.check_connection(conn)

In [ ]:
nb.list_schemas(conn)

## Check the runtime packages

Runs a small UDF that imports the training packages. On failure, run the command it prints and repeat.

In [ ]:
container_ok = nb.check_container(conn, alias=CONTAINER)

## Check MLflow

Shows where runs are recorded. Browser UI: `exakit mlflow start`, then `exakit mlflow url`. `MLFLOW_TRACKING_URI` overrides.

In [ ]:
mlflow_ok = nb.check_mlflow()

## Check the shared model directory

Tests that a UDF can read the kit's model directory (needed for Python model-file scoring). `exakit model list --json` shows its paths.

In [ ]:
bucket_ok = nb.check_bucket(conn)

## Choose a dataset

Start with the bundled sample: `SAMPLE = "esophageal_posts"` (social posts) or `"banking_complaints"` (complaints). It loads once and is reused. Keep `SOURCE = None` for this walkthrough.

In [ ]:
SAMPLE = "esophageal_posts"   # or "banking_complaints", or None
sample = nb.load_sample(conn, SAMPLE)

**Your own file (optional):** set `OWN_FILE` (CSV, Parquet or JSON) and `OWN_TABLE`. A missing path is skipped.

In [ ]:
OWN_FILE = "~/my-data/transactions.csv"   # .csv, .parquet or .json
OWN_TABLE = "MY_DATA.TRANSACTIONS"
own = nb.load_own_file(OWN_FILE, OWN_TABLE, conn=conn, missing_ok=True)

**A connected source (optional):** set `SOURCE = "SCHEMA.TABLE"` for a virtual-schema table. `None` skips it.

In [ ]:
SOURCE = None   # e.g. "VS_SALES.ORDERS"
source = nb.load_source(conn, SOURCE)

The data used from here on: the source, else your file, else the sample.

In [ ]:
ds = nb.first(source, own, sample)

## Approve the column roles

A **role** tells the notebooks how to use a column. The kit suggests roles with its reasons.

| Role | Meaning | Esophageal | Banking |
|---|---|---|---|
| `entity` | Who or what to compare | `SOURCE_DOMAIN` | `COMPANY` |
| `category` | Groups to compare | `SOCIAL_NETWORK` | `PRODUCT` |
| `date` | Event time | `START_DATE` | `DATE_RECEIVED` |
| `label` | Outcome to predict | `IS_NEGATIVE` | `MONETARY_RELIEF` |
| `text` | Free text | `MESSAGE` | none |

The **label** matters most: it is what a model learns to predict.

**Do now:** check the dropdowns, leave unrelated columns as **—**, click **Approve descriptor**, then run the next cell. Roles that are already approved need no click. Explore, Ask, Train and Share use this approved map.

Plain Python: `review.approve({"COLUMN": "role", ...})`.

In [ ]:
review = nb.review_roles(conn, ds)

In [ ]:
ds = review.dataset()
nb.show_roles(ds)

## Check the label

How many rows per group and the % where the label is 1. Label 1 means the outcome happened (a negative post, monetary relief), not that it is good. A note under the result says what the numbers mean.

In [ ]:
skipped = {m["role"]: m["reason"] for m in ds.missing}
for role, instead in (("category", "rows are counted per entity instead"),
                      ("label", "the label count is left out, and 01_train has nothing to predict until a label role is approved")):
    if role in skipped:
        print(f"No {role} role ({skipped[role]}): {instead}; the steps that need it later say so.")
BY = ds.col("category") if ds.has("category") else ds.col("entity")
GROUP, COUNT = nb.name(BY), nb.count_name(ds)       # columns named after the data: SOCIAL_NETWORK, POSTS
PCT = nb.name(ds.col("label"), "PCT") if ds.has("label") else None   # IS_NEGATIVE_PCT
RATE = f",\n       ROUND(100 * AVG({ds.col('label')}), 1) AS {PCT}" if PCT else ""
first_rows = nb.sql(conn, f"""
SELECT {BY} AS {GROUP},
       COUNT(*) AS {COUNT}{RATE}
FROM {ds.fact_table}
GROUP BY {BY}
ORDER BY {COUNT} DESC
LIMIT 10""")
nb.show_sql()
overall = nb.sql(conn, f"SELECT COUNT(*) AS N, ROUND(100 * AVG({ds.col('label')}), 1) AS PCT FROM {ds.fact_table}") if PCT else None
nb.note_groups(first_rows, ds, GROUP, COUNT, PCT, overall=(overall["N"][0], overall["PCT"][0]) if PCT else None)
first_rows

## Next

Run `data-scientist/01_explore.ipynb` up to its handoff cell, then open `01_train.ipynb`.